### Лабораторная работа №2. Реализация метода главных компонент (PCA)

#### Импорты

In [231]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression, Lasso
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
import plotly.graph_objects as go

#### Загрузка набора данных

In [175]:
data = load_breast_cancer()
data

{'data': array([[1.799e+01, 1.038e+01, 1.228e+02, ..., 2.654e-01, 4.601e-01,
         1.189e-01],
        [2.057e+01, 1.777e+01, 1.329e+02, ..., 1.860e-01, 2.750e-01,
         8.902e-02],
        [1.969e+01, 2.125e+01, 1.300e+02, ..., 2.430e-01, 3.613e-01,
         8.758e-02],
        ...,
        [1.660e+01, 2.808e+01, 1.083e+02, ..., 1.418e-01, 2.218e-01,
         7.820e-02],
        [2.060e+01, 2.933e+01, 1.401e+02, ..., 2.650e-01, 4.087e-01,
         1.240e-01],
        [7.760e+00, 2.454e+01, 4.792e+01, ..., 0.000e+00, 2.871e-01,
         7.039e-02]], shape=(569, 30)),
 'target': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0,
        0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0,
        1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0,
        1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1,
        1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1,

In [176]:
X = data.data
y = data.target

In [177]:
feature_names = data.feature_names

In [178]:
X.shape, y.shape

((569, 30), (569,))

In [179]:
data.target_names

array(['malignant', 'benign'], dtype='<U9')

In [180]:
X_df = pd.DataFrame(X, columns=feature_names)
X_df.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,radius error,texture error,perimeter error,area error,smoothness error,compactness error,concavity error,concave points error,symmetry error,fractal dimension error,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,1.0950,0.9053,8.589,153.40,0.006399,0.04904,0.05373,0.01587,0.03003,0.006193,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,0.5435,0.7339,3.398,74.08,0.005225,0.01308,0.01860,0.01340,0.01389,0.003532,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,0.7456,0.7869,4.585,94.03,0.006150,0.04006,0.03832,0.02058,0.02250,0.004571,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,0.4956,1.1560,3.445,27.23,0.009110,0.07458,0.05661,0.01867,0.05963,0.009208,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,0.7572,0.7813,5.438,94.44,0.011490,0.02461,0.05688,0.01885,0.01756,0.005115,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [181]:
X_df.describe()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,radius error,texture error,perimeter error,area error,smoothness error,compactness error,concavity error,concave points error,symmetry error,fractal dimension error,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
count,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000
mean,14.127292,19.289649,91.969033,654.889104,0.096360,0.104341,0.088799,0.048919,0.181162,0.062798,0.405172,1.216853,2.866059,40.337079,0.007041,0.025478,0.031894,0.011796,0.020542,0.003795,16.269190,25.677223,107.261213,880.583128,0.132369,0.254265,0.272188,0.114606,0.290076,0.083946
std,3.524049,4.301036,24.298981,351.914129,0.014064,0.052813,0.079720,0.038803,0.027414,0.007060,0.277313,0.551648,2.021855,45.491006,0.003003,0.017908,0.030186,0.006170,0.008266,0.002646,4.833242,6.146258,33.602542,569.356993,0.022832,0.157336,0.208624,0.065732,0.061867,0.018061
min,6.981000,9.710000,43.790000,143.500000,0.052630,0.019380,0.000000,0.000000,0.106000,0.049960,0.111500,0.360200,0.757000,6.802000,0.001713,0.002252,0.000000,0.000000,0.007882,0.000895,7.930000,12.020000,50.410000,185.200000,0.071170,0.027290,0.000000,0.000000,0.156500,0.055040
25%,11.700000,16.170000,75.170000,420.300000,0.086370,0.064920,0.029560,0.020310,0.161900,0.057700,0.232400,0.833900,1.606000,17.850000,0.005169,0.013080,0.015090,0.007638,0.015160,0.002248,13.010000,21.080000,84.110000,515.300000,0.116600,0.147200,0.114500,0.064930,0.250400,0.071460
50%,13.370000,18.840000,86.240000,551.100000,0.095870,0.092630,0.061540,0.033500,0.179200,0.061540,0.324200,1.108000,2.287000,24.530000,0.006380,0.020450,0.025890,0.010930,0.018730,0.003187,14.970000,25.410000,97.660000,686.500000,0.131300,0.211900,0.226700,0.099930,0.282200,0.080040
75%,15.780000,21.800000,104.100000,782.700000,0.105300,0.130400,0.130700,0.074000,0.195700,0.066120,0.478900,1.474000,3.357000,45.190000,0.008146,0.032450,0.042050,0.014710,0.023480,0.004558,18.790000,29.720000,125.400000,1084.000000,0.146000,0.339100,0.382900,0.161400,0.317900,0.092080
max,28.110000,39.280000,188.500000,2501.000000,0.163400,0.345400,0.426800,0.201200,0.304000,0.097440,2.873000,4.885000,21.980000,542.200000,0.031130,0.135400,0.396000,0.052790,0.078950,0.029840,36.040000,49.540000,251.200000,4254.000000,0.222600,1.058000,1.252000,0.291000,0.663800,0.207500


#### Реализация алгоритма PCA

In [182]:
def my_pca(X: np.ndarray, k: int):
    # Центрирование данных
    X_centered = X - X.mean(axis=0)
    # Построение ковариационной матрицы
    C = np.cov(X_centered, rowvar=False)
    # Диагонализация матрицы
    eigvals, eigvecs = np.linalg.eig(C)
    # Сортировка компонент
    sorted_indices = np.argsort(eigvals)[::-1]
    eigvals_sorted = eigvals[sorted_indices]
    eigvecs_sorted = eigvecs[:, sorted_indices]
    # Доля объясненной дисперсии
    explained_variance_ratio = eigvals_sorted / np.sum(eigvals_sorted)
    # Выбор K главных компонент
    W_K = eigvecs_sorted[:, :k]
    # Проекция данных
    Z = X_centered @ W_K
    # Возврат результата
    return Z, eigvals_sorted, explained_variance_ratio
    

#### Применение PCA

In [183]:
Z2_manual, exp_variance_manual, exp_variance_ratio_manual = my_pca(X, 2)
f'{Z2_manual.shape=}'

'Z2_manual.shape=(569, 2)'

In [184]:
pca_sk = PCA(n_components=2)
Z2_sklearn, exp_variance_sklearn, exp_variance_ratio_sklearn = pca_sk.fit_transform(X), pca_sk.explained_variance_, pca_sk.explained_variance_ratio_
f'{Z2_sklearn.shape=}'

'Z2_sklearn.shape=(569, 2)'

#### Сравнение реализаций

In [185]:
print(f'{exp_variance_manual[:2]=}')
print(f'{exp_variance_sklearn=}')
print(f'{exp_variance_ratio_manual[:2]=}')
print(f'{exp_variance_ratio_sklearn=}')

exp_variance_manual[:2]=array([443782.6051466 ,   7310.10006165])
exp_variance_sklearn=array([443782.6051466 ,   7310.10006165])
exp_variance_ratio_manual[:2]=array([0.98204467, 0.01617649])
exp_variance_ratio_sklearn=array([0.98204467, 0.01617649])


In [246]:
components_2 = [1, 2]

bar_char_fit = go.Figure()
bar_char_fit.add_trace(
    go.Bar(name='Z2_manual', x=components_2, y=exp_variance_ratio_manual[:2])
)
bar_char_fit.add_trace(
    go.Bar(name='Z2_sklearn', x=components_2, y=exp_variance_ratio_sklearn)
)
bar_char_fit.update_layout(
    barcornerradius=15,
    barmode='group'
)
bar_char_fit.update_layout(
    title='Доля объясненной дисперсии компонент по наборам данных',
    xaxis_title='Компонента',
    yaxis_title='Доля объясненной дисперсии',
    legend=dict(
        title='Набор данных:',
    ),
)
bar_char_fit.show()

In [245]:
scatter_plot_fig = go.Figure()
scatter_plot_fig.add_trace(
    go.Scatter(
        x=Z2_manual[:, 0],
        y=Z2_manual[:, 1],
        mode='markers',
        name='Z2_manual'
    )
)
scatter_plot_fig.add_trace(
    go.Scatter(
        x=Z2_sklearn[:, 0],
        y=Z2_sklearn[:, 1],
        mode='markers',
        name='Z2_sklearn'
    )
)
scatter_plot_fig.update_layout(
    title='Распределение значений двух главных компонент по наборам данных',
    xaxis_title='Значение 1 компоненты',
    yaxis_title='Значение 2 компоненты',
    legend=dict(
        title='Набор данных:',
    ),
)
scatter_plot_fig.show()

#### Выбор оптимального числа компонент

In [244]:
components = [i for i in range(1, len(exp_variance_ratio_manual) + 1)]

scree_plot_fig = go.Figure()
scree_plot_fig.add_trace(
    go.Bar(name='bars', x=components, y=exp_variance_ratio_manual)
)
scree_plot_fig.add_trace(
    go.Scatter(name='lines', x=components, y=exp_variance_ratio_manual, mode='lines+markers')
)
scree_plot_fig.update_layout(
    title='График осыпи (Scree plot)',
    xaxis_title='Номер компоненты',
    yaxis_title='Значение дисперсии',
    legend=dict(
        title='Значения:',
    ),
)
scree_plot_fig.show()

In [242]:
threshold = 0.999
cumulative = np.cumsum(exp_variance_ratio_manual)
threshold_line = [threshold for _ in range(len(exp_variance_ratio_manual))]

cumulative_fig = go.Figure()
cumulative_fig.add_trace(
    go.Scatter(name='Компонента', x=components, y=cumulative, mode='lines+markers')
)
cumulative_fig.add_trace(
    go.Scatter(name='Порог', x=components, y=threshold_line, mode='lines')
)
cumulative_fig.update_layout(
    title='График накопления объясненной дисперсии',
    xaxis_title='Номер компоненты',
    yaxis_title='Соотношение дисперсии',
    legend=dict(
        title='Значения:',
    ),
)
cumulative_fig.show()

In [240]:
malignant_val = np.array([v for i, v in enumerate(Z2_manual) if y[i] == 0])
benign_val = np.array([v for i, v in enumerate(Z2_manual) if y[i] == 1])

class_scatter_plot_fig = go.Figure() 
class_scatter_plot_fig.add_trace(
    go.Scatter(
        x=malignant_val[:, 0],
        y=malignant_val[:, 1],
        mode='markers',
        name='malignant'
    )
)
class_scatter_plot_fig.add_trace(
    go.Scatter(
        x=benign_val[:, 0],
        y=benign_val[:, 1],
        mode='markers',
        name='benign'
    )
)
class_scatter_plot_fig.update_layout(
    title='Распределение значений двух главных компонент по классам',
    xaxis_title='Значение 1 компоненты',
    yaxis_title='Значение 2 компоненты',
    legend=dict(
        title='Класс:',
    ),
)
class_scatter_plot_fig.show()

#### Эксперимент с Lasso

In [216]:
# Признаки должны быть стандартизованы для одинаковых штрафов относительно друг друга
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [238]:
alpha_range = np.arange(0.1, 0.5, 0.0001)
coefs = []

for alpha in alpha_range:
    l1_model = Lasso(alpha=alpha)
    l1_model.fit(X_scaled, y)
    coefs.append(l1_model.coef_)

coefs = np.array(coefs)

In [239]:
l1_coefs_fig = go.Figure()
for i in range(X.shape[1]):
    l1_coefs_fig.add_trace(
        go.Scatter(
            x=alpha_range,
            y=coefs[:, i],
            name=feature_names[i],
            mode='lines'
        )
    )
l1_coefs_fig.update_layout(
    title='Веса признаков в зависимости от степени l1-регуляризации',
    xaxis_title='Коэффициент l1-регуляризации',
    yaxis_title='Вес признака',
    legend=dict(
        title='Признак:',
    ),
)
l1_coefs_fig.show()

#### Обучение и сравнение моделей логистической регрессии 

In [193]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

In [194]:
unique_counts_train = np.unique_counts(y_train)
unique_counts_train

UniqueCountsResult(values=array([0, 1]), counts=array([145, 236]))

In [195]:
unique_counts_test = np.unique_counts(y_test)
unique_counts_test

UniqueCountsResult(values=array([0, 1]), counts=array([ 67, 121]))

In [211]:
bar_char_fit = go.Figure()
bar_char_fit.add_trace(
    go.Bar(name='Train', x=unique_counts_train.values, y=unique_counts_train.counts)
)
bar_char_fit.add_trace(
    go.Bar(name='Test', x=unique_counts_test.values, y=unique_counts_test.counts)
)
bar_char_fit.update_layout(
    title='Распределение классов по наборам данных',
    xaxis_title='Класс',
    yaxis_title='Количество элементов',
    legend=dict(
        title='Набор данных:',
    ),
    barcornerradius=15,
    barmode='group'
)
bar_char_fit.show()

In [197]:
# Для избежания утечки данных (data leakage)
new_pca_sk = PCA(n_components=2).fit(X_train)
X_train_pca = new_pca_sk.transform(X_train)
X_test_pca = new_pca_sk.transform(X_test)

In [198]:
iter_num = 10000
logreg = LogisticRegression(max_iter=iter_num)
logreg_pca = LogisticRegression(max_iter=iter_num)

In [199]:
logreg.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [200]:
logreg_pca.fit(X_train_pca, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [201]:
pred_logreg_train = logreg.predict(X_train)
pred_logreg_test = logreg.predict(X_test)
f1_logreg_train = f1_score(y_train, pred_logreg_train)
f1_logreg_test = f1_score(y_test, pred_logreg_test)

In [202]:
pred_logreg_pca_train = logreg_pca.predict(X_train_pca)
pred_logreg_pca_test = logreg_pca.predict(X_test_pca)
f1_logreg_pca_train = f1_score(y_train, pred_logreg_pca_train)
f1_logreg_pca_test = f1_score(y_test, pred_logreg_pca_test)

In [212]:
f1_list = [
    [f1_logreg_train, f1_logreg_test],
    [f1_logreg_pca_train, f1_logreg_pca_test]
]

heatmap_fig = go.Figure(
    go.Heatmap(
        x=['Train', 'Test'],
        y=['logreg', 'logreg_pca'],
        z=f1_list,
        text=np.round(f1_list, 3),
        texttemplate='%{text}',
        colorscale = 'ylgn',
    )
)
# Настраиваем макет
heatmap_fig.update_layout(
    title='Тепловая карта оценки F1-score (модель, набор данных)',
    xaxis_title='Набор данных',
    yaxis_title='Модель',
)
heatmap_fig.show()